<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_01_bronze_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · 01 · Ingesta batch → Bronze
Carga 4 maestros desde Landing a Bronze (Parquet particionado) con esquema explícito, columnas de linaje,
deduplicación por clave natural, quarantine de filas ilegibles y log de conteos.

In [9]:
!pip install -q "pyspark>=4.0,<4.1"
import os; os.environ.pop('JAVA_HOME', None)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
LAKE  = Path('/content/drive/MyDrive/cpa_datalake')
RUTAS = {z: LAKE / z for z in ['landing', 'stream_input', 'bronze', 'silver',
                               'gold', 'quarantine', 'checkpoints', 'logs']}

from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('cloud-provider-analytics')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
         .getOrCreate())
print('Spark', spark.version)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark 4.0.5


In [10]:
#esquemas explicitos
#particiones: los maestros son snapshots, una "foto" diaria, asi que se particionan por dia de ingesta. Billing se particiona por mes, su grano natural
S, D, B, DT = T.StringType(), T.DoubleType(), T.BooleanType(), T.DateType()
campo = lambda nombre, tipo: T.StructField(nombre, tipo, True)

ESQUEMAS = {
    'customers_orgs': T.StructType([
        campo('org_id', S), campo('org_name', S), campo('industry', S), campo('hq_region', S),
        campo('plan_tier', S), campo('is_enterprise', B), campo('signup_date', DT),
        campo('sales_rep', S), campo('lifecycle_stage', S), campo('marketing_source', S),
        campo('nps_score', D)]),
    'users': T.StructType([
        campo('user_id', S), campo('org_id', S), campo('email', S), campo('role', S),
        campo('active', B), campo('created_at', DT), campo('last_login', DT)]),
    'resources': T.StructType([
        campo('resource_id', S), campo('org_id', S), campo('service', S), campo('region', S),
        campo('created_at', DT), campo('state', S), campo('tags_json', S)]),
    'billing_monthly': T.StructType([
        campo('invoice_id', S), campo('org_id', S), campo('month', DT), campo('subtotal', D),
        campo('credits', D), campo('taxes', D), campo('currency', S),
        campo('exchange_rate_to_usd', D)]),
}

# clave natural para deduplicar y columna de partición de cada tabla
CONFIG = {
    'customers_orgs':  {'clave': 'org_id',      'particion': 'ingest_date'},
    'users':           {'clave': 'user_id',     'particion': 'ingest_date'},
    'resources':       {'clave': 'resource_id', 'particion': 'ingest_date'},
    'billing_monthly': {'clave': 'invoice_id',  'particion': 'month'},
}

In [11]:
#funcion de ingesta

#lo mas importante es la ultima verificacion: "cuadra" comprueba que {landing = bronze + invalidos + duplicados}.
#vendria a ser el objetivo "O3" de la entrega 1, ahora ejecutado.
#cada corrida queda registrada en "logs/ingesta_batch.jsonl", que es la evidencia de los logs que pide la consigna

#CORRECCION:
#. se agrego "option('escape', '"') ", esto le dice a spark que dentro de un campo entre comillas, " " significa una comilla simple.

import uuid, json, datetime

def ingerir_csv(nombre):
    cfg = CONFIG[nombre]
    run_id    = uuid.uuid4().hex[:8]
    ingest_ts = datetime.datetime.now(datetime.timezone.utc)
    esquema   = T.StructType(ESQUEMAS[nombre].fields + [campo('_corrupt_record', S)])

    crudo = (spark.read
             .option('header', True)
             .option('mode', 'PERMISSIVE')
             .option('columnNameOfCorruptRecord', '_corrupt_record')
             .option('escape', '"')          # CSV estándar: comillas internas duplicadas (""), en la primer corrida se detecto 211 filas invalidas por la comilla
             .schema(esquema)
             .csv(str(RUTAS['landing'] / f'{nombre}.csv'))
             .withColumn('source_file', F.col('_metadata.file_path'))     # linaje
             .withColumn('ingest_ts',   F.lit(ingest_ts))
             .withColumn('ingest_date', F.to_date('ingest_ts'))
             .withColumn('batch_id',    F.lit(run_id))
             .cache())

    invalidos = crudo.filter(F.col('_corrupt_record').isNotNull())
    validos   = crudo.filter(F.col('_corrupt_record').isNull()).drop('_corrupt_record')
    bronze    = validos.dropDuplicates([cfg['clave']])

    n = {'landing': crudo.count(), 'invalidos': invalidos.count(),
         'validos': validos.count(), 'bronze': bronze.count()}
    n['duplicados'] = n['validos'] - n['bronze']

    bronze.write.mode('overwrite').partitionBy(cfg['particion']).parquet(str(RUTAS['bronze'] / nombre))

    if n['invalidos']:
        (invalidos.select(F.lit('parse_error').alias('motivo'), F.lit('bronze').alias('zona_origen'),
                          'source_file', 'ingest_ts', 'ingest_date', '_corrupt_record')
         .write.mode('overwrite').partitionBy('ingest_date')
         .parquet(str(RUTAS['quarantine'] / nombre)))

    registro = {'tabla': nombre, 'run_id': run_id, 'ingest_ts': ingest_ts.isoformat(), **n,
                'cuadra': n['landing'] == n['bronze'] + n['invalidos'] + n['duplicados']}
    with open(RUTAS['logs'] / 'ingesta_batch.jsonl', 'a') as f:
        f.write(json.dumps(registro) + '\n')
    crudo.unpersist()
    return registro


In [12]:
import shutil
shutil.rmtree(RUTAS['quarantine'] / 'resources', ignore_errors=True)
print('Quarantine de resources limpiada: las 211 filas eran un error de lectura, no de datos')

Quarantine de resources limpiada: las 211 filas eran un error de lectura, no de datos


In [13]:
#ejecutar:
import pandas as pd
resultados = [ingerir_csv(t) for t in CONFIG]
pd.DataFrame(resultados)

,tabla,run_id,ingest_ts,landing,invalidos,validos,bronze,duplicados,cuadra
0,customers_orgs,7024307f,2026-11-15T18:10:48.688322+00:00,80,0,80,80,0,True
1,users,7c34413a,2026-11-15T18:10:49.870058+00:00,800,0,800,800,0,True
2,resources,42c08677,2026-11-15T18:10:51.396668+00:00,400,0,400,400,0,True
3,billing_monthly,2a4658e9,2026-11-15T18:10:52.755109+00:00,240,0,240,240,0,True


In [14]:
#verificar lo escrito en bronze
#"printSchema()" muestra que los tipos son los que declaramos (date, boolean, double) y no todo texto.
#"du" y "find" dan las rutas y tamaños, otras evidencia que pide la consigna
for t in CONFIG:
    df = spark.read.parquet(str(RUTAS['bronze'] / t))
    print(f'===== {t}: {df.count()} filas')
    df.printSchema()

!du -sh "{RUTAS['bronze']}"/*
!find "{RUTAS['bronze']}/billing_monthly" -maxdepth 1 -type d

spark.read.parquet(str(RUTAS['bronze'] / 'resources')).select('resource_id', 'tags_json').show(4, truncate=False)


===== customers_orgs: 80 filas
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- ingest_date: date (nullable = true)

===== users: 800 filas
root
 |-- user_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- role: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_login: date (nullable = true)
 |-- source_file: string (nullable

In [15]:
#prueba de idempotencia,si "antes" y "despues" son iguales, la re-ejecucion no duplica. es uno de los criterios de aceptacion de la consigna
antes = {t: spark.read.parquet(str(RUTAS['bronze'] / t)).count() for t in CONFIG}
[ingerir_csv(t) for t in CONFIG]                      # se re-ejecuta todo
despues = {t: spark.read.parquet(str(RUTAS['bronze'] / t)).count() for t in CONFIG}
pd.DataFrame({'antes': antes, 'después': despues})



,antes,después
customers_orgs,80,80
users,800,800
resources,400,400
billing_monthly,240,240


## cierre . ingesta batch -> Bronze

**que se hizo**

- se ingirieron 4 maestros desde landing a bronze en parquet: `customers_orgs`, `users`, `resources` y `billing_monthly`.
- **esquema explicito** por la tabla (fechas como `date`, flags como `boolean`, montos como `double`).
- **linaje** en cada fila : `source_file`, `ingest_ts`, `ingest_date` y `batch_id`.
- **particiones:** los maestros por `ingest_date` (snapshot diario); billing por `month` (`month=2025-06-01`, `2025-07-01`, `2025-08-01`).
- **deduplicacion** por clave natural y **quarantine** de filas ilegibles con `motivo = parse_error`.
- **control de conciliacion** por tabla: `landing = bronze + inválidos + duplicados`, registrado en `logs/ingesta_batch.jsonl`.

**hallazgo: la quarantine detecto un problema real de lectura**
en la primera corrida, 211 de 400 filas de `resources` fueron a quarantine.


**Causa:**`tags_json` contiene JSON con comas y comillas (`"[""env:prod"", ""pii:true""]"`), escapado con comillas duplicadas según el estándar CSV (RFC 4180). Spark espera por defecto `\"` como escape, así que no podía leer esas filas.
- **No se perdieron datos:** la conciliación cerró (`cuadra = True`) y el pipeline no se interrumpió.

**correccion:** `option('escape', '"')` en el lector de csv. se limpio la quarantine de esa corrida y se re-ingirio

sin esquema eexplicito y modo PERMISSIVE, essas filas se habrian cargado con las columnas desplazadas sin ningun aviso

**Resultados finales**
| Tabla | Landing | Inválidos | Duplicados | Bronze | ¿Cuadra? |
|---|---|---|---|---|---|
| customers_orgs | 80 | 0 | 0 | 80 | OK |
| users | 800 | 0 | 0 | 800 | OK |
| resources | 400 | 0 | 0 | 400 | OK |
| billing_monthly | 240 | 0 | 0 | 240 | OK |